> **First draft, no longer maintained.** Phase S is being rewritten from zero (see the Phase S page of the course). The material of this notebook moves into the new lessons; this version is kept only for reference.

# S1.1 · Centre, spread and shape

*Phase S, statistics and probability · S1, part 1 of 4 · about 2 hours*

Bibliographic data are rarely "normal". Reference lists, citation counts, author output and word frequencies are **right-skewed**: most values are small and a few are very large. The mean and the standard deviation, the summaries most of us learned first, describe such data badly. This lesson builds the toolkit that works: medians and percentiles, and three ways of drawing a distribution.

**Run this lesson:** [in your browser (JupyterLite)](https://sbrzt.github.io/personal-course/lite/lab/index.html?path=lessons/phase-s/first-draft/s1-1-centre-spread-shape.ipynb) · [in Google Colab](https://colab.research.google.com/github/sbrzt/personal-course/blob/main/lessons/phase-s/first-draft/s1-1-centre-spread-shape.ipynb) · on your computer (see *How this course works* on the course site)

**By the end you can**

- explain why the mean misleads on skewed data, and report the median and percentiles instead;
- say what a breakdown point is, and why it matters for scraped or OCR'd records;
- draw a distribution as a histogram, an ECDF and a CCDF, and say what each shows best;
- read the heavy tail of a citation distribution off a log–log plot.

## How the exercises work

Exercise cells are marked ✏️. Replace every `...` with your code and run the cell. The last line calls a check, which prints:

- ✅ when your answer is right, often with a note on what it means;
- ❌ with a hint when it is not right yet;
- ⬜ while something is still `...`.

Under each exercise there is a 💡 hint and a ✅ solution, both folded. Try for a few minutes before opening them: a little struggle is what makes this stick. The checks never stop the notebook, so you can skip an exercise and come back to it.

Run the setup cell first. In the browser it takes up to a minute, because it installs the course package.

In [ ]:
# ⚙️ Setup: run this cell first
import sys, pathlib, subprocess

if sys.platform == "emscripten":            # in the browser (JupyterLite)
    import piplite
    await piplite.install(["personal-course"])
elif "google.colab" in sys.modules:         # in Google Colab
    repo = "git+https://github.com/sbrzt/personal-course"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    f"personal-course @ {repo}"], check=True)
else:                                       # on your computer
    here = pathlib.Path.cwd().resolve()
    for folder in [here, *here.parents]:    # find the course folder
        if (folder / "course" / "__init__.py").exists():
            sys.path.insert(0, str(folder))
            break

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from course import data, plots, stats
from course.checks.draft import s1_1 as check

plots.use_style()
print("Ready.")

## 1 · A simulated journal

We start with simulated data, so we know exactly how they were made. `data.reference_counts()` returns the number of references in each of the 300 articles of an imaginary humanities journal. In S1.4 you run the same analysis on Bibliothecae\.it.

In [ ]:
refs = data.reference_counts()
print(len(refs), "articles")
print("first ten:", refs[:10])
print("smallest:", refs.min(), "   largest:", refs.max())

A first picture: a plain histogram.

In [ ]:
fig, ax = plt.subplots()
ax.hist(refs, bins=40, color=plots.BLUE, edgecolor="white", linewidth=0.6)
ax.set(xlabel="references in an article", ylabel="number of articles",
       title="Most articles cite 10 to 60 works; a few cite far more")
plt.show()

The bulk of the articles sits on the left, and a long tail stretches to the right. This shape is called **right-skewed** (or positively skewed): the tail points to the right.

## 2 · Mean versus median

The **mean** is the balance point: the sum divided by the count. The **median** is the middle value: half of the articles have fewer references, half have more. In symmetric data the two coincide. In right-skewed data the tail pulls the mean up, while the median stays with the typical article.

### ✏️ Exercise 1 · Mean versus median

Compute:

- `mean_refs`, the mean number of references;
- `median_refs`, the median;
- `share_above_mean`, the share of articles (a number between 0 and 1) with **more** references than the mean.

In [ ]:
mean_refs = ...
median_refs = ...
share_above_mean = ...

check.ex1(mean_refs, median_refs, share_above_mean)

<details>
<summary>💡 Hint</summary>

Use `np.mean` and `np.median`. For the share, `refs > mean_refs` gives an array of `True` and `False`; the mean of that array is the share of `True` values.

</details>

<details>
<summary>✅ Solution</summary>

```python
mean_refs = refs.mean()
median_refs = np.median(refs)
share_above_mean = np.mean(refs > mean_refs)

check.ex1(mean_refs, median_refs, share_above_mean)
```

The mean (39.9) is almost 30% higher than the median (31), and only 36% of the articles are "above average". When most units fall below the average, the average does not describe a typical unit.

</details>

### Robustness: one bad record

Bibliographic data are messy. A scraper can merge two reference lists, and an OCR error can turn 20 into 2000. How far can a single bad value move each summary?

### ✏️ Exercise 2 · One bad record

Add one article with 2,000 references, then compute how much the mean and the median **change**: `mean_shift` is the new mean minus the old one, and `median_shift` the new median minus the old one.

In [ ]:
refs_bad = np.append(refs, 2000)
mean_shift = ...
median_shift = ...

check.ex2(mean_shift, median_shift)

<details>
<summary>💡 Hint</summary>

Compute each statistic on `refs_bad` and on `refs`, then subtract.

</details>

<details>
<summary>✅ Solution</summary>

```python
refs_bad = np.append(refs, 2000)
mean_shift = refs_bad.mean() - refs.mean()
median_shift = np.median(refs_bad) - np.median(refs)

check.ex2(mean_shift, median_shift)
```

One record in 301 moved the mean by 6.5 references and left the median where it was.

</details>

The **breakdown point** of a statistic is the share of the data you must corrupt before you can make the statistic arbitrarily wrong. The mean's breakdown point is 0%: one value can drag it anywhere. The median's is 50%. With scraped or OCR'd records, prefer statistics with a high breakdown point, or clean the data first and say how you did it.

## 3 · Spread: percentiles instead of the standard deviation

The **standard deviation** (SD) measures spread around the mean, so it inherits the mean's weakness: it squares each deviation, and the squares of the tail values dominate the sum. **Percentiles** do not have this problem. The *p*th percentile is the value below which *p*% of the observations fall, so the median is the 50th percentile. The **interquartile range** (IQR) is the distance between the 25th and the 75th percentile: the spread of the middle half of the data.

In [ ]:
print(f"SD: {refs.std(ddof=1):.1f}")
print(f"SD with the bad record: {refs_bad.std(ddof=1):.1f}")
stats.summary(refs).round(1).to_frame("references")

### ✏️ Exercise 3 · Percentiles and the IQR

Compute `p90`, the 90th percentile of `refs`, and `iqr`, its interquartile range.

In [ ]:
p90 = ...
iqr = ...

check.ex3(p90, iqr)

<details>
<summary>💡 Hint</summary>

`np.percentile(refs, 90)` takes a percentage from 0 to 100, and `np.quantile(refs, 0.9)` takes a fraction from 0 to 1. Both give the same result.

</details>

<details>
<summary>✅ Solution</summary>

```python
p90 = np.percentile(refs, 90)
iqr = np.percentile(refs, 75) - np.percentile(refs, 25)

check.ex3(p90, iqr)
```

A good one-line description of skewed counts: "median 31 references (IQR 20–50); one article in ten cites more than 78 works; the longest list has 307." By default numpy interpolates between data points, and other software may use another rule, so percentiles can differ slightly between tools. When it matters, say which rule you used.

</details>

> **🔬 Metascience: the journal impact factor is a mean.** A journal's impact factor is the mean number of citations received by its recent articles. Citation counts are among the most skewed data there are, so a few highly cited papers set the value, and most articles in a journal receive fewer citations than its impact factor. This is one reason why the San Francisco Declaration on Research Assessment (DORA, 2012) advises against using journal-based metrics to assess individual articles or researchers, and why Larivière and colleagues (2016) proposed that journals publish their full citation distributions instead.

## 4 · Seeing the whole distribution

Summaries compress; plots show the shape. We now switch to more extreme data: the citations received by the 5,000 papers of a simulated research field. In S1.3 you will build the process that generated them.

In [ ]:
cites = data.citation_counts()
stats.summary(cites).round(2).to_frame("citations")

The mean is almost exactly 10. That is no accident: in this closed simulated field every paper cites 10 earlier papers, and every citation has two ends, so the mean number of citations received equals the mean number of references given. The median, 4, tells you about a typical paper.

A histogram on ordinary axes hides almost everything: one tall bar near zero and a tail too thin to see. Logarithmic axes help.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
axes[0].hist(cites, bins=50, color=plots.BLUE, edgecolor="white", linewidth=0.6)
axes[0].set(xlabel="citations", ylabel="number of papers",
            title="Linear axes: the tail is invisible")
plots.log_histogram(cites + 1, ax=axes[1], bins=25, color=plots.BLUE)
axes[1].set(xlabel="citations + 1", title="Log axes and log-spaced bins")
plt.tight_layout()
plt.show()

Two remarks on the right-hand plot. Zero cannot be placed on a log axis, so we plotted citations + 1, a common trick that you should always mention. And the bins get wider to the right, so the bars show a *density* (papers per unit of citations), not the number of papers in each bar.

A histogram also depends on how you choose its bins. Two plots avoid bins altogether:

- the **empirical cumulative distribution function** (ECDF) gives, for each value *x*, the share of observations at or below *x*;
- the **complementary cumulative distribution function** (CCDF) gives the share at or above *x*, written $P(X \ge x)$.

On log–log axes the CCDF shows the tail clearly, which is why it is the standard plot for citation data.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
x, f = stats.ecdf(cites)
axes[0].step(x, f, where="post", color=plots.BLUE)
axes[0].set(xscale="symlog", xlabel="citations (symmetric log scale)",
            ylabel="share of papers ≤ x", title="ECDF")
plots.ccdf(cites, ax=axes[1], color=plots.BLUE)
axes[1].set(xlabel="citations", ylabel="share of papers ≥ x",
            title="CCDF on log–log axes")
plt.tight_layout()
plt.show()

### ✏️ Exercise 4 · Your own CCDF

Write `my_ccdf(x)`. It returns two arrays: the distinct values of `x` in increasing order, and for each value *v* the share of observations at or above *v*. For `[1, 1, 2, 5]` it returns `[1, 2, 5]` and `[1.0, 0.5, 0.25]`.

In [ ]:
def my_ccdf(x):
    x = np.sort(np.asarray(x))
    values = ...
    shares = ...
    return values, shares

check.ex4(my_ccdf)

<details>
<summary>💡 Hint</summary>

`np.unique(x)` gives the distinct values in increasing order. For sorted `x`, `np.searchsorted(x, values, side="left")` counts, for each value, the observations smaller than it.

</details>

<details>
<summary>✅ Solution</summary>

```python
def my_ccdf(x):
    x = np.sort(np.asarray(x))
    values = np.unique(x)
    shares = 1 - np.searchsorted(x, values, side="left") / x.size
    return values, shares

check.ex4(my_ccdf)
```

`np.searchsorted(x, v, side="left")` counts the observations strictly below `v`, so one minus their share is the share at or above `v`. A loop such as `[np.mean(x >= v) for v in values]` also works, but it reads the whole array once for every distinct value, which gets slow on large data.

</details>

### ✏️ Exercise 5 · Reading the tail

Using `cites`, compute the share of papers that are never cited (`share_uncited`), that have at least 10 citations (`share_10_plus`), and that have at least 100 (`share_100_plus`). Then find the last two on the CCDF above.

In [ ]:
share_uncited = ...
share_10_plus = ...
share_100_plus = ...

check.ex5(share_uncited, share_10_plus, share_100_plus)

<details>
<summary>💡 Hint</summary>

As in Exercise 1: the mean of a boolean array is a share. "At least" means `>=`.

</details>

<details>
<summary>✅ Solution</summary>

```python
share_uncited = np.mean(cites == 0)
share_10_plus = np.mean(cites >= 10)
share_100_plus = np.mean(cites >= 100)

check.ex5(share_uncited, share_10_plus, share_100_plus)
```

On the CCDF, the point at 100 citations sits a little above 0.01: you can read such shares straight off the plot. The uncited papers do not appear at all, because a log axis has no zero. Always report the share of zeros separately: in bibliometrics, "uncitedness" is an indicator in its own right.

</details>

> **🤖 Why this matters for AI.** The same habits apply when you evaluate AI systems. An "average" accuracy, latency or cost per query hides the tail where systems fail: a model can look fine on average and be useless on the 5% of inputs you care about. Report medians and high percentiles (p90, p99), and plot the distribution. In S2 you will meet the skewed distribution that matters most for language models: the frequencies of words and tokens.

## 5 · Wrap-up

### ✍️ Explain it

In at most 100 words, explain to a colleague why "the average paper in this field receives 10 citations" misdescribes the typical paper, and say what you would report instead. Write your answer in the next cell.

*Your answer:*

<details>
<summary>A model answer</summary>

Citation counts are right-skewed: most papers receive few citations and a handful receive very many. The mean adds up all the citations and divides them by the number of papers, so the few highly cited papers pull it far above what a typical paper gets. In this simulated field the mean is 10, but the median is 4: half of the papers have 4 citations or fewer, and 18% have none. I would report the median with the interquartile range, the share of uncited papers and a high percentile, and show the whole distribution as a CCDF.

</details>

### Check your progress

In [ ]:
check.summary();

### Self-check

1. A dataset has mean 12 and median 4. Which way is it skewed, and what can you say about the share of values below the mean?
2. Why can a CCDF on log–log axes not show papers with zero citations, and what do you do about it?
3. Your scraper copied one article's 230 references five times. Which summaries can you still trust?

<details>
<summary>Answers</summary>

1. Right-skewed, because the mean is above the median. Since the mean is above the median, more than half of the values lie below the mean; how many more depends on the tail.
2. Zero has no place on a log axis (log 0 is undefined). Report the share of zeros separately, or plot x + 1 and say that you did.
3. The median and the IQR barely move. The mean, the SD and the maximum are no longer reliable.

</details>

### Done when

- [ ] Exercises 1 to 5 show ✅ in `check.summary()`.
- [ ] You can explain the breakdown point in one sentence.
- [ ] You have written your 100-word explanation.

### Further reading

- Downey, A. B. *Think Stats*, 3rd edition, chapters 1–4. O'Reilly. Free online: <https://allendowney.github.io/ThinkStats/>
- Larivière, V., Kiermer, V., MacCallum, C. J., et al. (2016). A simple proposal for the publication of journal citation distributions. *bioRxiv*, 062109. <https://doi.org/10.1101/062109>
- DORA (2012). *San Francisco Declaration on Research Assessment*. <https://sfdora.org/read/>
- Hicks, D., Wouters, P., Waltman, L., de Rijcke, S., & Rafols, I. (2015). Bibliometrics: The Leiden Manifesto for research metrics. *Nature*, 520, 429–431. <https://doi.org/10.1038/520429a>

**Next:** S1.2 · Concentration and association.